**01 data preprocessing & feature engineering**

**OBJECTIVE**

Clean the Nassau Candy Distributor data, engineer shipping and profitability features, and create the modeling dataset.


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
DATA=Path("Nassau Candy Distributor.csv")
df=pd.read_csv(DATA)
df.columns=[c.strip() for c in df.columns]
df.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


In [ ]:
df.shape, df.info(), df.isna().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  object 
 2   Order Date      10194 non-null  object 
 3   Ship Date       10194 non-null  object 
 4   Ship Mode       10194 non-null  object 
 5   Customer ID     10194 non-null  int64  
 6   Country/Region  10194 non-null  object 
 7   City            10194 non-null  object 
 8   State/Province  10194 non-null  object 
 9   Postal Code     10194 non-null  object 
 10  Division        10194 non-null  object 
 11  Region          10194 non-null  object 
 12  Product ID      10194 non-null  object 
 13  Product Name    10194 non-null  object 
 14  Sales           10194 non-null  float64
 15  Units           10194 non-null  int64  
 16  Gross Profit    10194 non-null  float64
 17  Cost            10194 non-null 

((10194, 18),
 None,
 Row ID            0
 Order ID          0
 Order Date        0
 Ship Date         0
 Ship Mode         0
 Customer ID       0
 Country/Region    0
 City              0
 State/Province    0
 Postal Code       0
 Division          0
 Region            0
 Product ID        0
 Product Name      0
 Sales             0
 Units             0
 Gross Profit      0
 Cost              0
 dtype: int64)

In [ ]:
df=df.drop_duplicates().copy()
df["Order Date"]=pd.to_datetime(df["Order Date"],errors="coerce")
df["Ship Date"]=pd.to_datetime(df["Ship Date"],errors="coerce")
df["Lead_Time"]=(df["Ship Date"]-df["Order Date"]).dt.days
df["Profit Margin"]=np.where(df["Sales"]!=0,df["Gross Profit"]/df["Sales"],np.nan)
df["Cost Percentage"]=np.where(df["Sales"]!=0,df["Cost"]/df["Sales"],np.nan)
df["Unit Price"]=np.where(df["Units"]!=0,df["Sales"]/df["Units"],np.nan)
df.describe(include="all").T


/tmp/ipykernel_1009/2077254951.py:3: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Ship Date"]=pd.to_datetime(df["Ship Date"],errors="coerce")


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Row ID,10194.0,NaN,NaN,NaN,5097.5,1.0,2549.25,5097.5,7645.75,10194.0,2942.898656
Order ID,10194,8549,US-2022-130974-CHO-SCR-58000,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,4130,NaN,NaN,NaN,2025-01-25 22:38:45.617433344,2024-01-02 00:00:00,2024-08-09 00:00:00,2025-02-11 00:00:00,2025-07-12 00:00:00,2025-12-12 00:00:00,NaN
Ship Date,10194,NaN,NaN,NaN,2028-10-23 23:20:43.790465024,2026-06-30 00:00:00,2027-11-09 00:00:00,2028-12-18 00:00:00,2029-11-08 00:00:00,2030-06-28 00:00:00,NaN
Ship Mode,10194,4,Standard Class,6120,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,10194.0,NaN,NaN,NaN,134468.961154,100006.0,117212.0,133550.0,152051.0,192314.0,20231.483007
Country/Region,10194,2,United States,9994,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,10194,542,New York City,915,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State/Province,10194,59,California,2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Postal Code,10194,654,10035,263,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
factory_map={
"Wonka Bar - Nutty Crunch Surprise":"Lot's O' Nuts","Wonka Bar - Fudge Mallows":"Lot's O' Nuts",
"Wonka Bar -Scrumdiddlyumptious":"Lot's O' Nuts","Wonka Bar - Milk Chocolate":"Wicked Choccy's",
"Wonka Bar - Triple Dazzle Caramel":"Wicked Choccy's","Laffy Taffy":"Sugar Shack",
"SweeTARTS":"Sugar Shack","Nerds":"Sugar Shack","Fun Dip":"Sugar Shack",
"Fizzy Lifting Drinks":"Sugar Shack","Everlasting Gobstopper":"Secret Factory",
"Hair Toffee":"The Other Factory","Lickable Wallpaper":"Secret Factory",
"Wonka Gum":"Secret Factory","Kazookles":"The Other Factory"}
df["Factory"]=df["Product Name"].map(factory_map)
df.to_csv("/content/Nassau Candy Distributor.csv",index=False)
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Product Name,Sales,Units,Gross Profit,Cost,Lead_Time,Profit Margin,Cost Percentage,Unit Price,Factory
0,1,US-2021-103800-CHO-MIL-31000,2024-03-01,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,851.0,0.649231,0.350769,3.25,Wicked Choccy's
1,2,US-2021-112326-CHO-TRI-54000,2024-04-01,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,821.0,0.653333,0.346667,3.75,Wicked Choccy's
2,3,US-2021-112326-CHO-NUT-13000,2024-04-01,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,821.0,0.713467,0.286533,3.49,Lot's O' Nuts
3,4,US-2021-112326-CHO-SCR-58000,2024-04-01,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,821.0,0.694444,0.305556,3.60,Lot's O' Nuts
4,5,US-2021-141817-CHO-TRI-54000,2024-05-01,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,795.0,0.653333,0.346667,3.75,Wicked Choccy's
